# The Water Bill - Rice x Water-Stress Summary

*Women in Data "What's Cooking?" Datathon, Sept 2026. Built by Heidi, porting the pipeline shape from
`discovery/Barley_Water_Risk_Simulator.ipynb` onto the rice production/trade work in
`discovery/QCL_all_Rice_Data_code28.ipynb` (Shruti & Heidi). See `PROJECT_WORKFLOW.md` at the repo root
for how every notebook in this repo relates to this one and to the team's to-do list.*

**The deliverable.** For a rice-importing country, how exposed is its supply to water stress in the
countries it buys from, and what happens to that supply if a stressed supplier's exports drop? This
notebook builds the four pieces Daniela's Sept 7 problem statement named "The Water Bill":

1. **Water exposure** - what share of a country's rice imports come from water-stressed exporters.
2. **Supplier concentration** - how many suppliers an importer actually relies on (HHI).
3. **Shock result** - what a production cut at a stressed supplier does to an importer's supply.
4. **Alternative suppliers** - who else could fill the gap, ranked by spare capacity and water headroom.

**Data sources and vintage** (all FAOSTAT bulk downloads, refreshed each run): QCL (Production, Crops and
Livestock Products), TCL (Trade, Crops and Livestock Products), the Detailed Trade Matrix (bilateral
reporter/partner trade), and SDGB (SDG Indicators, series 6.4.2 - freshwater withdrawal as % of available
resources). Production and trade figures cover 2000-2024; water-stress figures are each country's most
recent reported year, which varies by country (see the credibility-check section below).

**Before this ships**: every cell needs a `Run All` with output saved (this notebook was authored and
code-reviewed in an environment where FAOSTAT's bulk-download domain is blocked by network policy, so it
has not yet been run end to end - do that in Colab before the final push, per `GIT_RUNBOOK.md`).


## Section 0: Setup

In [ ]:
# Section 0: Setup
import urllib.request, zipfile, io, os, json, textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


In [ ]:
# Everything this notebook saves (charts + tables) goes into one outputs/ folder, so a
# single download after Run All brings back everything worth committing to the repo
# alongside this notebook (see PROJECT_WORKFLOW.md).
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Section 0: Data principles, encoded as config so every downstream calc references the same
# assumptions instead of re-deciding them inline.
# Rice-side values carried over verbatim from QCL_all_Rice_Data_code28.ipynb's CONFIG (Shruti & Heidi);
# water-stress values carried over from Barley_Water_Risk_Simulator.ipynb's CONFIG.
CONFIG = {
    # --- Production (QCL) ---
    'crop_item_code': 27,          # Rice, paddy (QCL) -- production is NOT double-counted with
                                    # milled forms; see docs/WiD_CodesForRiceAndConfusion.docx
    'crop_name': 'Rice',

    # --- Trade (TCL / TM) ---
    # Rice production (QCL code 27, paddy basis) is not what crosses borders -- trade is reported in
    # milled forms, so exports need converting back to a paddy-equivalent basis before comparing to
    # production. See docs/WiD_CodesForRiceAndConfusion.docx and the FAO conversion-factor PDF
    # (page 723): https://openknowledge.fao.org/server/api/core/bitstreams/63b8626c-7019-4f8c-a961-e975e7f9e285/content
    'trade_crop_item_code_rm': 31,       # milled/white rice -> paddy-equivalent
    'trade_crop_name_rm': 'Rice; milled',
    'rm_paddy_factor': 0.67,
    'trade_crop_item_code_rmh': 28,      # husked/brown rice -> paddy-equivalent
    'trade_crop_name_rmh': 'Husked rice',
    'rmh_paddy_factor': 0.77,
    # Do NOT add codes 29 (child of 28), 32 (child of 31), or 30 (FAO's own milled-equivalent
    # accumulator, wrong basis for a direct add to 27) -- see docs/WiD_CodesForRiceAndConfusion.docx.

    # --- Area/country cleanup ---
    'aggregate_code_cutoff': 5000,       # FAOSTAT regional/continental aggregates are >= 5000
    'exclude_area_codes': [351, 357, 214, 96, 128],   # China variants -- keep mainland (41) only

    # --- Reporter shortlisting (Section 2) ---
    'reexport_threshold_pct': 130,       # export share of production above this = re-exporter, not a
                                          # genuine source (see Section 2)
    'shortlist_min_export_t': 1_000_000, # minimum paddy-eq tonnes exported to be considered a
                                          # meaningful reporter for the Trade Matrix pull
    'shortlist_min_export_ratio_pct': 5, # minimum export-share-of-production to count as
                                          # export-oriented (arbitrary threshold, flagged for revisit)

    # --- Water stress (SDG 6.4.2: freshwater withdrawal as % of available freshwater resources) ---
    'water_stress_bins':   [0, 25, 50, 75, 100, float('inf')],
    'water_stress_labels': ['No stress', 'Low stress', 'Medium stress', 'High stress', 'Critical'],
    'stress_threshold_pct': 25,          # UN-Water's own threshold for entering water stress at all

    # --- Shock simulator (Section: Shock Result) ---
    # Elasticity-1 assumption: a pct_reduction cut in a supplier's production is modeled as the same
    # pct cut in what they export to a given importer. No substitution effects, no price response.
    # Diversification assumes unconstrained headroom at alternative suppliers. Both restated in the
    # Limitations section at the end of this notebook.
}
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'water_stress_bins'}, indent=2, default=str))


## Section 1: Rice production baseline
(adapted from `QCL_all_Rice_Data_code28.ipynb` Section 1, Shruti & Heidi)

In [ ]:
# Section 1a: QCL bulk download (Production: Crops and livestock products)
BULK_URL = 'https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
if not os.path.exists('qcl_bulk.zip'):
    print('Downloading QCL bulk file...')
    urllib.request.urlretrieve(BULK_URL, 'qcl_bulk.zip')

with zipfile.ZipFile('qcl_bulk.zip') as z:
    csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    print(f'Reading {csv_name}...')
    qcl_all = pd.read_csv(z.open(csv_name), encoding='latin-1', low_memory=False)
    item_csv = [n for n in z.namelist() if 'ItemCodes' in n][0]
    item_codes = pd.read_csv(z.open(item_csv), encoding='latin-1')
    area_csv = [n for n in z.namelist() if 'AreaCodes' in n][0]
    area_codes = pd.read_csv(z.open(area_csv), encoding='latin-1')

print(f'Full QCL: {len(qcl_all):,} rows')


In [ ]:
# Section 1b: filter to Rice Production (paddy, tonnes), drop regional/continental aggregates
# (< 5000) AND the China aggregate/territory codes (keeping mainland China only, per CONFIG).
qcl_rice = qcl_all[(qcl_all['Item Code'] == CONFIG['crop_item_code']) & (qcl_all['Element'] == 'Production')].copy()
print(f"Rice, Production element: {len(qcl_rice):,} rows, unit(s): {qcl_rice['Unit'].unique()}")

area_code_exclude = area_codes[
    (area_codes['Area Code'] < CONFIG['aggregate_code_cutoff'])
    & (~area_codes['Area Code'].isin(CONFIG['exclude_area_codes']))
]
valid_country_codes = set(area_code_exclude['Area Code'].astype(str))
qcl_rice = qcl_rice[qcl_rice['Area Code'].astype(str).isin(valid_country_codes)].copy()
print(f"After dropping aggregates + China double-counts: {len(qcl_rice):,} rows, {qcl_rice['Area'].nunique()} countries")

rice_by_area_year = qcl_rice.groupby(['Area', 'Area Code', 'Year'])['Value'].sum().reset_index(name='production_t')

rice_production_2000_2024 = (
    rice_by_area_year[(rice_by_area_year['Year'] >= 2000) & (rice_by_area_year['Year'] <= 2024)]
    .groupby('Area')['production_t'].sum().reset_index()
)
rice_production_2000_2024['pct_share_production'] = (
    rice_production_2000_2024['production_t'] / rice_production_2000_2024['production_t'].sum() * 100
)
rice_production_2000_2024 = rice_production_2000_2024.sort_values('production_t', ascending=False).reset_index(drop=True)
rice_production_2000_2024['rank'] = rice_production_2000_2024.index + 1
print(rice_production_2000_2024.head(15))


In [ ]:
# Section 1c: TCL bulk download (Trade: Crops and livestock products) -- exports, lifted to
# paddy-equivalent using the two conversion factors above (this is the "layer 2" approach the team
# validated against code30's cross-check; see docs/WiD_CodesForRiceAndConfusion.docx).
TCL_URL = 'https://bulks-faostat.fao.org/production/Trade_CropsLivestock_E_All_Data_(Normalized).zip'
if not os.path.exists('tcl_bulk.zip'):
    print('Downloading TCL bulk file...')
    urllib.request.urlretrieve(TCL_URL, 'tcl_bulk.zip')

with zipfile.ZipFile('tcl_bulk.zip') as z:
    tcl_csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                    and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    tcl_all = pd.read_csv(z.open(tcl_csv_name), encoding='latin-1', low_memory=False)
    tcl_area_csv = [n for n in z.namelist() if 'AreaCodes' in n][0]
    tcl_area_codes = pd.read_csv(z.open(tcl_area_csv), encoding='latin-1')

print(f'Full TCL: {len(tcl_all):,} rows')


In [ ]:
# Section 1d: filter TCL to the two rice trade items, Export quantity (tonnes), drop aggregates,
# and lift each milled form to paddy-equivalent with its own factor before summing.
rm_code,  rm_factor  = CONFIG['trade_crop_item_code_rm'],  CONFIG['rm_paddy_factor']
rmh_code, rmh_factor = CONFIG['trade_crop_item_code_rmh'], CONFIG['rmh_paddy_factor']

tcl_rice = tcl_all[
    (tcl_all['Item Code'].isin([rm_code, rmh_code])) & (tcl_all['Element'] == 'Export quantity')
].copy()
print(f"Rice trade (export): {len(tcl_rice):,} rows, unit(s): {tcl_rice['Unit'].unique()}")

tcl_rice['paddy_factor'] = tcl_rice['Item Code'].map({rm_code: rm_factor, rmh_code: rmh_factor})
tcl_rice['exported_paddy_eq_t'] = tcl_rice['Value'] / tcl_rice['paddy_factor']

area_code_exclude_tcl = tcl_area_codes[
    (tcl_area_codes['Area Code'] < CONFIG['aggregate_code_cutoff'])
    & (~tcl_area_codes['Area Code'].isin(CONFIG['exclude_area_codes']))
]
valid_country_codes_tcl = set(area_code_exclude_tcl['Area Code'].astype(str))
tcl_rice = tcl_rice[tcl_rice['Area Code'].astype(str).isin(valid_country_codes_tcl)].copy()
print(f"After dropping aggregates + China double-counts: {len(tcl_rice):,} rows, {tcl_rice['Area'].nunique()} countries")

rice_by_area_year_traded_conversion_based = (
    tcl_rice.groupby(['Area', 'Area Code', 'Year'])['exported_paddy_eq_t'].sum().reset_index(name='exported_t')
)

rice_exports_eq_2000_2024 = (
    rice_by_area_year_traded_conversion_based[
        (rice_by_area_year_traded_conversion_based['Year'] >= 2000)
        & (rice_by_area_year_traded_conversion_based['Year'] <= 2024)
    ].groupby('Area')['exported_t'].sum().reset_index()
)
rice_exports_eq_2000_2024['pct_share_export'] = (
    rice_exports_eq_2000_2024['exported_t'] / rice_exports_eq_2000_2024['exported_t'].sum() * 100
)
rice_exports_eq_2000_2024 = rice_exports_eq_2000_2024.sort_values('exported_t', ascending=False).reset_index(drop=True)
rice_exports_eq_2000_2024['rank'] = rice_exports_eq_2000_2024.index + 1
print(rice_exports_eq_2000_2024.head(15))


In [ ]:
# Section 1e: merge production + export shares -- the production-vs-export scatter this team
# already validated (India, Thailand, Vietnam, Pakistan as the standout exporters relative to
# production share).
rice_production_export = rice_production_2000_2024[
    ['Area', 'production_t', 'pct_share_production', 'rank']
].merge(
    rice_exports_eq_2000_2024[['Area', 'exported_t', 'pct_share_export', 'rank']],
    on='Area', how='outer', suffixes=('_production', '_export')
).rename(columns={'rank_production': 'production_rank', 'rank_export': 'export_rank'}).sort_values(
    'production_t', ascending=False
)
print(rice_production_export.head(20))


In [ ]:
# Reproduce the production-vs-export scatter (top 10 producers) as a sanity check that this
# session's downloads match what Shruti & Heidi found originally.
plot_data = rice_production_export.sort_values('production_t', ascending=False).head(10)

plt.figure(figsize=(10, 7))
plt.scatter(plot_data['pct_share_production'], plot_data['pct_share_export'])
for _, row in plot_data.iterrows():
    plt.annotate(row['Area'], (row['pct_share_production'], row['pct_share_export']),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)
max_value = max(plot_data['pct_share_production'].max(), plot_data['pct_share_export'].max())
plt.plot([0, max_value], [0, max_value], linestyle='--', label='export share = production share')
plt.xlabel('Share of Global Production (%)')
plt.ylabel('Share of Global Exports (%)')
plt.title('Rice: Production vs Export Share, 2000-2024')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rice_production_vs_export.png', dpi=150)
plt.show()


## Section 2: Which countries matter for trade? Export dependence & reporter shortlisting
(adapted from `QCL_all_Rice_Data_code28.ipynb` Section 3, Shruti & Heidi)

In [ ]:
# Separate real producers from re-exporters (trade a token amount they don't grow, e.g. Saudi
# Arabia, South Africa) and inactive countries, then flag DISGUISED re-exporters -- producers whose
# export share of their own production is implausibly high.
trade_df = rice_production_export.copy()
REEXPORT_THRESHOLD = CONFIG['reexport_threshold_pct']

producers   = trade_df[trade_df['production_t'] > 0].copy()
reexporters = trade_df[(trade_df['production_t'] == 0) & (trade_df['exported_t'] > 0)].copy()
inactive    = trade_df[(trade_df['production_t'] == 0) & (trade_df['exported_t'] == 0)].copy()

producers['export_ratio_of_production'] = producers['exported_t'] / producers['production_t'] * 100
disguised = producers[producers['export_ratio_of_production'] > REEXPORT_THRESHOLD].copy()
disguised['note'] = 're-exporter (token production, exports >> output)'
producers = producers[producers['export_ratio_of_production'] <= REEXPORT_THRESHOLD].copy()

reexporters['export_ratio_of_production'] = pd.NA
reexporters['note'] = 're-exporter (no domestic production)'
reexporters = pd.concat([reexporters, disguised], ignore_index=True)
inactive['export_ratio_of_production'] = pd.NA
inactive['note'] = 'neither produces nor exports'

producers['category'] = 'producer'
reexporters['category'] = 're-exporter'
inactive['category'] = 'inactive'
trade_classified = pd.concat([producers, reexporters, inactive], ignore_index=True).drop(columns='note', errors='ignore')

print(trade_classified['category'].value_counts())
print(f"total countries: {len(trade_classified)}")


In [ ]:
# Only genuine producers that are real, export-oriented exporters go into the Trade Matrix pull --
# this IS the "REPORTERS" list, computed dynamically rather than hardcoded, so it stays correct if
# the underlying FAOSTAT data is revised.
shortlist = trade_classified[
    (trade_classified['category'] == 'producer')
    & (trade_classified['exported_t'] > CONFIG['shortlist_min_export_t'])
    & (trade_classified['export_ratio_of_production'] > CONFIG['shortlist_min_export_ratio_pct'])
].sort_values('exported_t', ascending=False)

REPORTERS = shortlist['Area'].tolist()
print(shortlist[['Area', 'exported_t', 'export_ratio_of_production', 'pct_share_export', 'export_rank']].to_string(index=False))
print(f"\n{len(REPORTERS)} reporters selected for the Trade Matrix pull")


## Section 3: Trade Matrix - reporter to partner rice flows
(adapted from `QCL_all_Rice_Data_code28.ipynb` Section 3, Shruti & Heidi)

In [ ]:
# Detailed Trade Matrix bulk download -- large file, may take several minutes. Filtered inline to
# only the reporters shortlisted above, the two rice trade item codes, Export quantity, 2000-2024.
TM_BULK = 'https://bulks-faostat.fao.org/production/Trade_DetailedTradeMatrix_E_All_Data_(Normalized).zip'
if not os.path.exists('tm_bulk.zip'):
    print('Downloading TM bulk (~400 MB zip / ~1.5 GB unzipped)...')
    urllib.request.urlretrieve(TM_BULK, 'tm_bulk.zip')

ITEM_CODES = [rm_code, rmh_code]
ELEMENTS = [5910]   # Export quantity (reporter's exports TO each partner)
YEAR_MIN, YEAR_MAX = 2000, 2024

with zipfile.ZipFile('tm_bulk.zip') as z:
    def is_meta(n):
        low = n.lower().replace('_', '')
        return any(k in low for k in ('flags', 'itemcodes', 'elements', 'countrycodes', 'areacodes'))
    tm_csv_name = [n for n in z.namelist() if n.endswith('.csv') and not is_meta(n)][0]
    print(f'Streaming {tm_csv_name}...')

    parts = []
    for chunk in pd.read_csv(z.open(tm_csv_name), encoding='latin-1', low_memory=False, chunksize=1_000_000):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk['Item Code'] = pd.to_numeric(chunk['Item Code'], errors='coerce')
        chunk['Element Code'] = pd.to_numeric(chunk['Element Code'], errors='coerce')
        chunk['Year'] = pd.to_numeric(chunk['Year'], errors='coerce')
        m = (chunk['Reporter Countries'].isin(REPORTERS) & chunk['Item Code'].isin(ITEM_CODES)
             & chunk['Element Code'].isin(ELEMENTS) & chunk['Year'].between(YEAR_MIN, YEAR_MAX))
        if m.any():
            parts.append(chunk.loc[m])

    if not parts:
        raise ValueError("No rows matched -- check reporter spelling / item code coverage.")
    tm_rice = pd.concat(parts, ignore_index=True)

found = set(tm_rice['Reporter Countries'].unique())
missing = [r for r in REPORTERS if r not in found]
print(f"rows: {len(tm_rice):,}   reporters found: {len(found)}/{len(REPORTERS)}")
if missing:
    print("!! not found (check spelling in the TM file):", missing)


In [ ]:
# Total export quantity from each reporter to each partner, then each partner's dependency on
# each reporter as a share of what it buys FROM THE REPORTER SET (not the partner's total imports
# from the whole world -- we only have the shortlisted reporters' data).
flows = (tm_rice.groupby(['Reporter Countries', 'Partner Countries'])['Value'].sum()
         .reset_index(name='export_qty_t'))
flows = flows[flows['export_qty_t'] > 0].copy()

partner_totals = (flows.groupby('Partner Countries')['export_qty_t'].sum()
                   .reset_index(name='partner_total_from_reporters'))
dependency = flows.merge(partner_totals, on='Partner Countries')
dependency['dependency_pct'] = (dependency['export_qty_t'] / dependency['partner_total_from_reporters'] * 100).round(1)
dependency = dependency.sort_values(['Partner Countries', 'dependency_pct'], ascending=[True, False]).reset_index(drop=True)

dependency.to_csv(f'{OUTPUT_DIR}/Reporter_Partner_Export_Dependency_28_31.csv', index=False)
print(dependency.head(20).to_string(index=False))


## [TODO-1 | Water Exposure] Join rice reporter/partner countries to SDG 6.4.2 water stress
(adapted from `Barley_Water_Risk_Simulator.ipynb` Section 3-4; rice flows from `QCL_all_Rice_Data_code28.ipynb`
Section 3, Shruti & Heidi)

In [ ]:
# SDG bulk download (SDGB) -- same source used in the barley template.
SDG_BULK = 'https://bulks-faostat.fao.org/production/SDG_BulkDownloads_E_All_Data_(Normalized).zip'
if not os.path.exists('sdgb_bulk.zip'):
    print('Downloading SDGB bulk file...')
    urllib.request.urlretrieve(SDG_BULK, 'sdgb_bulk.zip')

with zipfile.ZipFile('sdgb_bulk.zip') as z:
    sdgb_csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                     and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    sdgb_all = pd.read_csv(z.open(sdgb_csv_name), encoding='latin-1', low_memory=False)
    flags_lookup = pd.read_csv(z.open('SDG_BulkDownloads_E_Flags.csv'), encoding='latin-1')

water_stress_all = sdgb_all[sdgb_all['Item Code'] == '24027-_T'].copy()  # headline SDG 6.4.2, no sub-breakdown
print(f'{len(water_stress_all):,} rows, {water_stress_all["Area Code"].nunique()} areas, '
      f'{water_stress_all["Year"].min()}-{water_stress_all["Year"].max()}')

official_flags = flags_lookup[flags_lookup[' Description'].str.contains('official|reported', case=False, na=False)]['Flag'].tolist()


In [ ]:
# Take the LATEST available value per country (reporting is patchy -- see caveat in Section 0/
# Limitations) rather than a fixed reference year, and never impute a missing reading.
water_stress_all = water_stress_all.dropna(subset=['Value']).copy()
water_stress_all['is_official'] = water_stress_all['Flag'].isin(official_flags)

idx_latest = water_stress_all.groupby('Area Code')['Year'].idxmax()
water_stress_latest = water_stress_all.loc[idx_latest, ['Area', 'Area Code', 'Year', 'Value', 'Flag', 'is_official']]
water_stress_latest = water_stress_latest.rename(columns={'Year': 'ws_year', 'Value': 'water_stress_pct'})
water_stress_latest['water_stress_pct'] = pd.to_numeric(water_stress_latest['water_stress_pct'], errors='coerce')
water_stress_latest = water_stress_latest.dropna(subset=['water_stress_pct']).copy()

water_stress_latest['water_stress_tier'] = pd.cut(
    water_stress_latest['water_stress_pct'], bins=CONFIG['water_stress_bins'],
    labels=CONFIG['water_stress_labels'], right=False)
water_stress_latest['is_stressed'] = water_stress_latest['water_stress_pct'] >= CONFIG['stress_threshold_pct']

print(f"Countries with any water-stress reading: {len(water_stress_latest)}")
print(water_stress_latest['water_stress_tier'].value_counts().sort_index().to_string())


In [ ]:
# Join the REPORTER (exporter) side of each flow to its own water-stress reading. Matched by
# country NAME, not code -- the Trade Matrix's Reporter/Partner columns only carry names, and since
# both TM and SDGB are FAOSTAT domains they should share the same official spellings. We check for
# and print any names that DON'T match rather than silently dropping them (same discipline as the
# barley template's 'No data' handling).
NAME_ALIASES = {
    # left: name as it may appear in TM/QCL/TCL  ->  right: name as it appears in SDGB, if different.
    # Populate this if the match report below flags any mismatches; empty until proven necessary.
}

def match_water_stress(names):
    aliased = pd.Series(names).map(lambda n: NAME_ALIASES.get(n, n))
    ws = water_stress_latest.set_index('Area')
    matched = aliased.isin(ws.index)
    return aliased, matched

reporter_names = dependency['Reporter Countries'].unique()
aliased_reporters, matched_mask = match_water_stress(reporter_names)
unmatched_reporters = sorted(set(pd.Series(reporter_names)[~matched_mask]))
print(f"Reporters matched to a water-stress reading: {matched_mask.sum()}/{len(reporter_names)}")
if unmatched_reporters:
    print("!! NOT matched (add to NAME_ALIASES above once you confirm the SDGB spelling):")
    print(unmatched_reporters)


In [ ]:
# Attach reporter-side water stress to every flow. Rows for an unmatched/no-data reporter are
# kept and labeled 'No data', not dropped -- so 'not yet known' stays visibly different from
# 'confirmed not stressed'.
ws_lookup = water_stress_latest[['Area', 'ws_year', 'water_stress_pct', 'water_stress_tier', 'is_stressed']].copy()
ws_lookup = ws_lookup.rename(columns={'Area': 'Reporter Countries'})

dependency_ws = dependency.copy()
dependency_ws['Reporter Countries'] = dependency_ws['Reporter Countries'].map(lambda n: NAME_ALIASES.get(n, n))
dependency_ws = dependency_ws.merge(ws_lookup, on='Reporter Countries', how='left')
dependency_ws['water_stress_tier'] = dependency_ws['water_stress_tier'].astype(object).fillna('No data')
dependency_ws['is_stressed'] = dependency_ws['is_stressed'].fillna(False)

cols = ['Partner Countries', 'Reporter Countries', 'export_qty_t', 'dependency_pct',
        'water_stress_pct', 'water_stress_tier', 'ws_year']
print(dependency_ws.sort_values('dependency_pct', ascending=False).head(25)[cols].to_string(index=False))


In [ ]:
# Per-importer (partner) exposure: what share of THIS country's rice imports (from our reporter
# set) comes from water-stressed suppliers, plus concentration metrics reused for the next section.
def summarize_importer(df):
    total = df['export_qty_t'].sum()
    d = df.assign(share=df['export_qty_t'] / total).sort_values('share', ascending=False)
    top = d.iloc[0]
    return pd.Series({
        'total_export_from_reporters_t': total,
        'n_suppliers': len(d),
        'top_supplier': top['Reporter Countries'],
        'top_supplier_share_pct': top['share'] * 100,
        'hhi': (d['share'] ** 2).sum() * 10000,   # Herfindahl-Hirschman Index, 0-10000 scale
        'pct_from_stressed_suppliers': d.loc[d['is_stressed'], 'share'].sum() * 100,
        'pct_from_no_data_suppliers': d.loc[d['water_stress_tier'] == 'No data', 'share'].sum() * 100,
    })

importer_exposure = (dependency_ws.groupby('Partner Countries').apply(summarize_importer, include_groups=False)
                      .reset_index().rename(columns={'Partner Countries': 'importer'})
                      .sort_values('pct_from_stressed_suppliers', ascending=False).reset_index(drop=True))

print("Most import-exposed rice-buying countries (share of rice imports from water-stressed suppliers):")
print(importer_exposure.head(20)[['importer', 'total_export_from_reporters_t', 'n_suppliers',
                                   'top_supplier', 'top_supplier_share_pct', 'hhi',
                                   'pct_from_stressed_suppliers']].to_string(index=False))


In [ ]:
top_exp = importer_exposure.head(15)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_exp['importer'], top_exp['pct_from_stressed_suppliers'], color='#c0392b')
ax.set_ylabel('% of rice imports from water-stressed suppliers')
ax.set_title('Most import-exposed rice-buying countries')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rice_import_exposure.png', dpi=150)
plt.show()


## [TODO-2 | Credibility check] Daniela's claimed stats vs. what this notebook computes
Daniela's Sept 7 problem statement (`docs/datathon_initial_ideas_09072026.docx`) named specific numbers
for the video. Before those numbers go in front of judges, this section checks them against what this
notebook actually computes from FAOSTAT data -- flagging matches and mismatches explicitly rather than
silently trusting or silently overwriting either one.

In [ ]:
# Daniela's claimed numbers, transcribed verbatim from her Sept 7 write-up. These are INPUTS to the
# check below, not validated facts -- treat every row's 'status' as the answer, not this table.
claimed = pd.DataFrame([
    {'metric': 'Pakistan water withdrawal %',      'country': 'Pakistan',    'claimed_value': 107,  'unit': '% (SDG 6.4.2)'},
    {'metric': 'Pakistan rice water volume',        'country': 'Pakistan',    'claimed_value': 4.97, 'unit': 'km3'},
    {'metric': 'Pakistan export share (of ? )',     'country': 'Pakistan',    'claimed_value': 53,   'unit': '%'},
    {'metric': 'Afghanistan dependency on Pakistan','country': 'Afghanistan','claimed_value': 98,   'unit': '% dependency_pct'},
    {'metric': 'Kazakhstan dependency',             'country': 'Kazakhstan', 'claimed_value': 86,   'unit': '% dependency_pct'},
    {'metric': 'Kenya dependency',                  'country': 'Kenya',      'claimed_value': 67,   'unit': '% dependency_pct'},
    {'metric': 'Egypt water withdrawal %',          'country': 'Egypt',      'claimed_value': 113,  'unit': '% (SDG 6.4.2)'},
    {'metric': 'Egypt import dependency',           'country': 'Egypt',      'claimed_value': 83,   'unit': '% dependency_pct'},
])

def computed_water_stress(country):
    row = water_stress_latest[water_stress_latest['Area'] == NAME_ALIASES.get(country, country)]
    return None if row.empty else float(row.iloc[0]['water_stress_pct'])

def computed_dependency_on_pakistan(importer):
    row = dependency_ws[(dependency_ws['Partner Countries'] == importer)
                         & (dependency_ws['Reporter Countries'] == 'Pakistan')]
    return None if row.empty else float(row.iloc[0]['dependency_pct'])

results = []
for _, r in claimed.iterrows():
    if 'withdrawal' in r['metric']:
        computed = computed_water_stress(r['country'])
    elif 'dependency' in r['metric']:
        computed = computed_dependency_on_pakistan(r['country'])
    else:
        computed = None   # water volume (km3) and export-share-of-what needs a source Daniela hasn't
                           # specified yet -- not something this pipeline currently derives
    if computed is None:
        status = 'NOT YET COMPUTED -- needs a data source or definition from Daniela'
    elif abs(computed - r['claimed_value']) <= max(3, r['claimed_value'] * 0.1):
        status = 'MATCH (within 10%)'
    else:
        status = f'MISMATCH -- flag before the video'
    results.append({**r, 'computed_value': computed, 'status': status})

credibility_check = pd.DataFrame(results)
print(credibility_check.to_string(index=False))


**Read this table before recording anything.** Anything marked `NOT YET COMPUTED` means this
notebook has no basis for that number yet -- either the metric needs a data source this pipeline
doesn't pull (e.g. a water-volume-per-tonne footprint figure, which is a Mekonnen & Hoekstra-style
calculation this notebook doesn't yet do), or Daniela's phrasing needs a tighter definition (e.g.
"export share" of what denominator) before it can be checked. `MISMATCH` rows need a conversation
before they go in the video, not a silent edit in either direction.

## [TODO-3 | Supplier Concentration] Concentration index per importer
(built on the existing `dependency_pct` computed in Section 3, HHI pattern adapted from
`Barley_Water_Risk_Simulator.ipynb` Section 6)

In [ ]:
# HHI (Herfindahl-Hirschman Index, 0-10000) is already computed per importer in importer_exposure
# above. US antitrust guidance treats >2500 as "highly concentrated" -- useful, ready-made framing for
# the video since it's a threshold judges/viewers outside this field can look up themselves.
concentration = importer_exposure.copy()
concentration['concentration_label'] = pd.cut(
    concentration['hhi'], bins=[0, 1500, 2500, 10001],
    labels=['Unconcentrated', 'Moderately concentrated', 'Highly concentrated'], right=False)

print(concentration.sort_values('hhi', ascending=False).head(20)
      [['importer', 'n_suppliers', 'top_supplier', 'top_supplier_share_pct', 'hhi', 'concentration_label']]
      .to_string(index=False))


In [ ]:
top_hhi = concentration.sort_values('hhi', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(11, 5))
colors = top_hhi['hhi'].apply(lambda h: '#c0392b' if h > 2500 else ('#e67e22' if h > 1500 else '#2c7fb8'))
ax.bar(top_hhi['importer'], top_hhi['hhi'], color=colors)
ax.axhline(2500, linestyle='--', color='gray', linewidth=1)
ax.text(len(top_hhi) - 1, 2550, 'highly concentrated threshold', ha='right', fontsize=8, color='gray')
ax.set_ylabel('Herfindahl-Hirschman Index (rice import suppliers)')
ax.set_title('Most supplier-concentrated rice-buying countries')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rice_supplier_concentration.png', dpi=150)
plt.show()


## [TODO-4 | Shock Result] Pakistan supply-shock scenario
(`simulate_shock()` ported verbatim from `Barley_Water_Risk_Simulator.ipynb` Section 7; applied here to
Pakistan specifically, since Pakistan is the stressed supplier Daniela's narrative centers on -- not
auto-selected, so this stays the country the video is actually about even if the data would otherwise
pick a different "most exposed" pair.)

In [ ]:
# Pick the importer most dependent on Pakistan specifically, among Pakistan's actual buyers in our
# data -- this is what the credibility-check table above is trying to verify against Afghanistan/
# Kazakhstan/Kenya. Override SELECTED_IMPORTER manually below if the video should feature a different
# one of the three.
SHOCK_SUPPLIER = 'Pakistan'
pakistan_buyers = (dependency_ws[dependency_ws['Reporter Countries'] == SHOCK_SUPPLIER]
                    .sort_values('dependency_pct', ascending=False))
print("Pakistan's buyers, by dependency on Pakistan specifically:")
print(pakistan_buyers[['Partner Countries', 'export_qty_t', 'dependency_pct']].head(10).to_string(index=False))

SELECTED_IMPORTER = pakistan_buyers.iloc[0]['Partner Countries']
print(f'\nPrototype country: {SELECTED_IMPORTER}')

supplier_mix = (dependency_ws[dependency_ws['Partner Countries'] == SELECTED_IMPORTER]
                [['Reporter Countries', 'export_qty_t', 'water_stress_tier', 'is_stressed']]
                .rename(columns={'Reporter Countries': 'supplier'})
                .sort_values('export_qty_t', ascending=False).reset_index(drop=True))
supplier_mix['share_pct'] = supplier_mix['export_qty_t'] / supplier_mix['export_qty_t'].sum() * 100
print(f"\n{SELECTED_IMPORTER}'s full supplier mix (from our reporter set):")
print(supplier_mix.to_string(index=False))


In [ ]:
def simulate_shock(mix, supplier, pct_reduction, diversify=False):
    """
    Elasticity-1 assumption: a pct_reduction cut in `supplier`'s production is modeled as the same
    pct cut in what they export to this importer (no substitution effects, no price response --
    see Limitations). If diversify=True, the lost volume is reallocated proportionally across the
    importer's OTHER existing suppliers, in proportion to their current share, assuming unconstrained
    headroom (a stated simplification, see Limitations).
    """
    m = mix.copy()
    lost = m.loc[m['supplier'] == supplier, 'export_qty_t'].iloc[0] * pct_reduction
    m.loc[m['supplier'] == supplier, 'export_qty_t'] -= lost

    if diversify:
        others = m['supplier'] != supplier
        other_total = m.loc[others, 'export_qty_t'].sum()
        if other_total > 0:
            m.loc[others, 'export_qty_t'] += lost * (m.loc[others, 'export_qty_t'] / other_total)

    new_total = m['export_qty_t'].sum()
    baseline_total = mix['export_qty_t'].sum()
    gap_pct = (baseline_total - new_total) / baseline_total * 100
    pct_stressed_after = (m.loc[m['is_stressed'], 'export_qty_t'].sum() / new_total * 100
                           if new_total > 0 else np.nan)
    return {'gap_pct': gap_pct, 'pct_from_stressed_after': pct_stressed_after, 'mix': m}


scenario_results = []
for pct in [0.10, 0.20, 0.30]:
    no_div = simulate_shock(supplier_mix, SHOCK_SUPPLIER, pct, diversify=False)
    with_div = simulate_shock(supplier_mix, SHOCK_SUPPLIER, pct, diversify=True)
    scenario_results.append({
        'reduction_pct': int(pct * 100),
        'gap_no_diversification_pct': no_div['gap_pct'],
        'gap_with_diversification_pct': with_div['gap_pct'],
        'pct_from_stressed_no_div': no_div['pct_from_stressed_after'],
        'pct_from_stressed_with_div': with_div['pct_from_stressed_after'],
    })

scenario_df = pd.DataFrame(scenario_results)
print(scenario_df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scenario_df['reduction_pct'], scenario_df['gap_no_diversification_pct'], marker='o', label='No diversification')
ax.plot(scenario_df['reduction_pct'], scenario_df['gap_with_diversification_pct'], marker='o', label='With diversification')
ax.set_xlabel(f'{SHOCK_SUPPLIER} rice export cut (%)')
ax.set_ylabel(f"{SELECTED_IMPORTER}'s resulting rice supply gap (%)")
ax.set_title(f'Shock scenario: {SELECTED_IMPORTER} sourcing from {SHOCK_SUPPLIER}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rice_shock_scenario.png', dpi=150)
plt.show()


## [TODO-5 | Alternative Suppliers] Diversification ranking
(ported from `Barley_Water_Risk_Simulator.ipynb` Section 7's `diversify=True` path)

In [ ]:
# Rank the selected importer's OTHER existing suppliers (excluding Pakistan) by two things a
# real diversification decision would weigh: how much spare capacity they already show (current
# share, as a proxy -- an existing relationship is easier to scale than a brand-new one) and how much
# water headroom they have (lower water-stress tier is better).
alt_suppliers = supplier_mix[supplier_mix['supplier'] != SHOCK_SUPPLIER].copy()

tier_order = {label: i for i, label in enumerate(CONFIG['water_stress_labels'] + ['No data'])}
alt_suppliers['water_headroom_rank'] = alt_suppliers['water_stress_tier'].map(tier_order)
alt_suppliers = alt_suppliers.sort_values(
    ['water_headroom_rank', 'share_pct'], ascending=[True, False]).reset_index(drop=True)
alt_suppliers['diversification_rank'] = alt_suppliers.index + 1

print(f"Alternative suppliers for {SELECTED_IMPORTER}, ranked by water headroom then existing share:")
print(alt_suppliers[['diversification_rank', 'supplier', 'share_pct', 'water_stress_tier']].to_string(index=False))


## Video-ready exports
Every chart and table above saves into a single `outputs/` folder inside this Colab session (created in Section 0). After `Run All`:

1. In Colab's left-hand file panel, find the `outputs` folder, right-click it, and choose **Download**. This gets you one zip with all 4 PNGs and all 4 CSVs.
2. Unzip it into `summary/outputs/` in your local `seed-and-scale` clone (create that folder if it doesn't exist yet).
3. Save this notebook itself back to the same path via **File > Save a copy in GitHub** (same repo, same branch, same `summary/WiD_Summary_TheWaterBill.ipynb` path) so the run notebook with its populated cell outputs overwrites the code-only version already on `wid_summary`.
4. `git pull` locally, then `git add summary/outputs summary/WiD_Summary_TheWaterBill.ipynb && git commit` -- notebook and exported files land in the same commit, and there's now a single history of what the notebook produced at each run.

This gives Daniela both a rendered notebook to screen-share from GitHub AND standalone PNGs she can drop straight into a slide or the video without touching Colab at all.

In [ ]:
importer_exposure.to_csv(f'{OUTPUT_DIR}/the_water_bill_importer_exposure.csv', index=False)
credibility_check.to_csv(f'{OUTPUT_DIR}/the_water_bill_credibility_check.csv', index=False)
scenario_df.to_csv(f'{OUTPUT_DIR}/the_water_bill_pakistan_shock_scenario.csv', index=False)
alt_suppliers.to_csv(f'{OUTPUT_DIR}/the_water_bill_alternative_suppliers.csv', index=False)

print(f'Exported to {OUTPUT_DIR}/:')
for fname in ['the_water_bill_importer_exposure.csv', 'the_water_bill_credibility_check.csv',
              'the_water_bill_pakistan_shock_scenario.csv', 'the_water_bill_alternative_suppliers.csv',
              'rice_production_vs_export.png', 'rice_import_exposure.png',
              'rice_supplier_concentration.png', 'rice_shock_scenario.png']:
    print(f'  {OUTPUT_DIR}/{fname}')


## Limitations & next steps
*(mirrors the pattern in `Barley_Water_Risk_Simulator.ipynb` Section 9)*

**What's verified:**
- Production and export figures use the conversion-factor logic the team validated in
  `WiD_CodesForRiceAndConfusion.docx` (QCL code 27 for production; TCL codes 28÷0.77 + 31÷0.67 for
  exports, cross-checked against code30's own milled-equivalent accumulator in a separate notebook).
- The reporter shortlist (Section 2) is computed dynamically from the data, not hardcoded, so it
  stays correct as FAOSTAT revises figures.
- Water-stress values are each country's LATEST available SDG 6.4.2 reading -- never imputed; a
  country with no reading is labeled "No data" and excluded from stressed-supplier percentages,
  not silently assumed safe.

**What's estimated or simplified:**
- The shock simulator assumes 1:1 elasticity (a supplier's production cut translates directly and
  fully into an export cut to every buyer) and no price response -- real supply shocks trigger price
  changes and substitution effects this doesn't model.
- The diversification scenario assumes unconstrained headroom at alternative suppliers -- in
  practice, scaling up a secondary supplier has its own limits (land, water, logistics) not modeled
  here.
- Country-name matching between the Trade Matrix and SDGB (Section: Water Exposure) relies on
  FAOSTAT's naming being consistent across domains. The match-report cell prints anything that
  didn't match; if `NAME_ALIASES` is still empty after a real run, that's confirmed, not assumed.
- This notebook does NOT yet compute a water-VOLUME figure (e.g. Daniela's claimed "4.97 km3 rice
  water" for Pakistan) -- that would need a water-footprint-per-tonne figure (Mekonnen & Hoekstra
  style) multiplied by production/export tonnage, which is a different data source than anything
  pulled in so far. Flagged in the credibility-check table above as NOT YET COMPUTED.

**Still open (TODO-6/7 from `PROJECT_WORKFLOW.md`):**
- TODO-6: a country-selector widget so anyone can pick a different importer live in the video,
  instead of the notebook running a single hardcoded scenario.
- TODO-7: the video script and recording itself (Daniela).
- Any `MISMATCH` or `NOT YET COMPUTED` row from the credibility-check table needs a conversation with
  Daniela before it goes in front of judges.
